# HAI Data Cleaning

**Cleans and reshapes the raw HAI training data into participant-level wide format.**
- Input: `train_hai.tsv` (long format — one row per participant × timepoint × strain)
- Output: `hai_cleaned.csv` (wide format — one row per participant, log2 titer per strain-timepoint column)

## Design notes

**Log2 transform:** Applied to all titer values. Consistent with downstream modeling notebooks which operate in log2 space.

**Timepoint mapping:** Day 30 is aliased to Day 28 (group means are nearly identical). Only days 0, 28, and 365 are retained.

**Pivot:** Long rows are pivoted to wide format; column names follow `HAI_{strain}_d{day}`.

In [11]:
TIMEPOINTS_TO_KEEP = [0.0, 28.0, 365.0]
TIMEPOINT_ALIAS = {30.0: 28.0}

In [12]:
DATA_PATH = '../../data'
CLEAN_DATA_PATH = '../../cleaned_data'

In [13]:
import os

import numpy as np
import pandas as pd

In [14]:
df = pd.read_csv(DATA_PATH + '/train_hai.tsv', sep='\t')
print(f'Raw shape: {df.shape}')
df.head()

Raw shape: (128177, 6)


,hai_id,participant_id,timepoint,virus_strain,value,material
0,ID_001__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_001,0.0,H1N1 A/South Carolina/1/1918,20.,Unknown
1,ID_001__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_001,28.0,H1N1 A/South Carolina/1/1918,40.,Unknown
2,ID_002__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_002,0.0,H1N1 A/South Carolina/1/1918,5.,Unknown
3,ID_002__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_002,28.0,H1N1 A/South Carolina/1/1918,5.,Unknown
4,ID_003__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_003,0.0,H1N1 A/South Carolina/1/1918,80.,Unknown


### Drop unused columns, filter invalid strains, and coerce values

In [15]:
df = df.drop(columns=['hai_id', 'material'])
df = df[df['virus_strain'] != '-']
df['value'] = pd.to_numeric(df['value'], errors='coerce')
print(f'After dropping unused columns and filtering invalid strains: {df.shape}')

After dropping unused columns and filtering invalid strains: (127959, 4)


### Timepoint analysis

In [16]:
df['timepoint'].value_counts().sort_index()

timepoint
-7.0         18
 0.0      49017
 3.0        318
 7.0        136
 14.0       777
 28.0     48240
 30.0       468
 70.0       180
 75.0       318
 90.0      2202
 180.0       51
 365.0    23808
Name: count, dtype: int64

In [17]:
df.groupby('timepoint')['value'].mean()

timepoint
-7.0       47.777778
 0.0       73.792527
 3.0      160.424528
 7.0      208.851544
 14.0      28.403520
 28.0     142.261548
 30.0     141.481744
 70.0     149.000000
 75.0     317.751572
 90.0     137.563579
 180.0    144.705882
 365.0     83.095474
Name: value, dtype: float64

### Filter and alias timepoints

In [18]:
df['timepoint'] = df['timepoint'].replace(TIMEPOINT_ALIAS)
df = df[df['timepoint'].isin(TIMEPOINTS_TO_KEEP)]
print(f'After timepoint filtering: {df.shape}')
df.head()

After timepoint filtering: (121533, 4)


,participant_id,timepoint,virus_strain,value
0,2016_UGA.ID_001,0.0,H1N1 A/South Carolina/1/1918,20.0
1,2016_UGA.ID_001,28.0,H1N1 A/South Carolina/1/1918,40.0
2,2016_UGA.ID_002,0.0,H1N1 A/South Carolina/1/1918,5.0
3,2016_UGA.ID_002,28.0,H1N1 A/South Carolina/1/1918,5.0
4,2016_UGA.ID_003,0.0,H1N1 A/South Carolina/1/1918,80.0


### Pivot to wide format and apply log2 transform

In [19]:
df['hai_timepoint'] = 'HAI_' + df['virus_strain'].astype(str) + '_d' + df['timepoint'].astype(int).astype(str)

df_pivot = df.pivot_table(
    index='participant_id',
    columns='hai_timepoint',
    values='value'
)

df_pivot = df_pivot.reset_index()
df_pivot = df_pivot.rename_axis(None, axis=1)

for col in df_pivot.columns:
    if col != 'participant_id':
        df_pivot[col] = df_pivot[col].apply(lambda x: np.log2(x) if pd.notna(x) else x)

print(f'Pivoted shape: {df_pivot.shape}')
df_pivot

Pivoted shape: (3757, 196)


,participant_id,HAI_Anc B/Lee/1940_d0,HAI_Anc B/Lee/1940_d28,HAI_Anc B/Lee/1940_d365,HAI_Anc B/Maryland/1959_d0,HAI_Anc B/Maryland/1959_d28,HAI_Anc B/Singapore/1964_d0,HAI_Anc B/Singapore/1964_d28,HAI_H1N1 A/Beijing/262/1995_d0,HAI_H1N1 A/Beijing/262/1995_d28,...,HAI_Yam B/Sichuan/379/1999_d365,HAI_Yam B/Texas/6/2011_d0,HAI_Yam B/Texas/6/2011_d28,HAI_Yam B/Texas/6/2011_d365,HAI_Yam B/Wisconsin/1/2010_d0,HAI_Yam B/Wisconsin/1/2010_d28,HAI_Yam B/Wisconsin/1/2010_d365,HAI_Yam B/Yamagata/16/1988_d0,HAI_Yam B/Yamagata/16/1988_d28,HAI_Yam B/Yamagata/16/1988_d365
0,2016_UGA.ID_001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.321928,5.321928,...,NaN,7.321928,8.321928,NaN,8.321928,8.321928,NaN,7.321928,8.321928,NaN
1,2016_UGA.ID_002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.321928,5.321928,...,NaN,5.321928,6.321928,NaN,6.321928,6.321928,NaN,5.321928,5.321928,NaN
2,2016_UGA.ID_003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.321928,5.321928,...,NaN,8.321928,8.321928,NaN,8.321928,8.321928,NaN,8.321928,8.321928,NaN
3,2016_UGA.ID_004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.321928,5.321928,...,NaN,4.321928,6.321928,NaN,5.321928,6.321928,NaN,3.321928,5.321928,NaN
4,2016_UGA.ID_005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.321928,2.321928,...,6.321928,6.321928,6.321928,5.321928,6.321928,6.321928,5.321928,5.321928,5.321928,5.321928
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3752,SDY887.SUB134259,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3753,SDY887.SUB134260,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3754,SDY887.SUB197783,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3755,SDY887.SUB197784,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
os.makedirs(CLEAN_DATA_PATH, exist_ok=True)
df_pivot.to_csv(CLEAN_DATA_PATH + '/hai_cleaned.csv', index=False)
print(f'Saved to {CLEAN_DATA_PATH}/hai_cleaned.csv')

Saved to ../../cleaned_data/hai_cleaned.csv
